# 14. Outlier Detection — Statistical, Distance & Machine Learning Methods

Outliers are observations that deviate markedly from the remaining population. Distinguishing between genuine anomalies (fraud, sensor failure) and natural heavy-tailed extremes is critical in EDA.

---

### Outlier Detection Spectrum
1. **Univariate Methods**:
   - **Tukey's IQR Rule**: Inner fences at $[Q_1 - 1.5 \times IQR, Q_3 + 1.5 \times IQR]$.
   - **Standard Z-Score**: Flags points where $|Z| = \left|\frac{x - \mu}{\sigma}\right| > 3.0$.
   - **Modified Z-Score**: Replaces mean/std with median and MAD (Median Absolute Deviation).
2. **Multivariate Methods**:
   - **Isolation Forest**: Subsamples data and isolates points using random partitioning trees.
   - **Local Outlier Factor (LOF)**: Measures local density deviation relative to $k$-nearest neighbors.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

np.random.seed(42)
n_inliers = 950
n_outliers = 50

# Synthesize 2D continuous distribution
inliers = np.random.multivariate_normal(mean=[50, 50], cov=[[10, 5], [5, 12]], size=n_inliers)
outliers = np.random.uniform(low=0, high=100, size=(n_outliers, 2))

X = np.vstack([inliers, outliers])
df_outliers = pd.DataFrame(X, columns=['feature_1', 'feature_2'])
print(f"Total Observations: {len(df_outliers)} (Known inliers: {n_inliers}, Injected anomalies: {n_outliers})")

Total Observations: 1000 (Known inliers: 950, Injected anomalies: 50)

## 1. Univariate Statistical Detection: IQR vs Modified Z-Score

In [1]:
def detect_iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

def detect_modified_zscore_outliers(series, threshold=3.5):
    med = series.median()
    mad = np.median(np.abs(series - med))
    mod_z = 0.6745 * (series - med) / (mad + 1e-9)
    return np.abs(mod_z) > threshold

iqr_flags = detect_iqr_outliers(df_outliers['feature_1'])
mod_z_flags = detect_modified_zscore_outliers(df_outliers['feature_1'])

print("=== UNIVARIATE OUTLIER DETECTION REPORT ===")
print(f"Tukey's IQR Outlier Count (Feature 1):        {iqr_flags.sum()} ({iqr_flags.mean()*100:.2f}%)")
print(f"Modified Z-Score Outlier Count (Feature 1):   {mod_z_flags.sum()} ({mod_z_flags.mean()*100:.2f}%)")

=== UNIVARIATE OUTLIER DETECTION REPORT ===
Tukey's IQR Outlier Count (Feature 1):        42 (4.20%)
Modified Z-Score Outlier Count (Feature 1):   39 (3.90%)

## 2. Multivariate Detection: Isolation Forest & LOF

In [1]:
iso_forest = IsolationForest(contamination=0.05, random_state=42)
iso_preds = iso_forest.fit_predict(X)
iso_outliers = (iso_preds == -1)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_preds = lof.fit_predict(X)
lof_outliers = (lof_preds == -1)

overlap = np.sum(iso_outliers & lof_outliers)

print("=== MULTIVARIATE ANOMALY DETECTION REPORT ===")
print(f"Isolation Forest Detected Outliers: {np.sum(iso_outliers)}")
print(f"Local Outlier Factor (LOF) Outliers: {np.sum(lof_outliers)}")
print(f"Consensus Agreement (Both Models):  {overlap} anomalies confirmed by both detectors")

=== MULTIVARIATE ANOMALY DETECTION REPORT ===
Isolation Forest Detected Outliers: 50
Local Outlier Factor (LOF) Outliers: 50
Consensus Agreement (Both Models):  44 anomalies confirmed by both detectors